In [ ]:
import os
import json
from metadata.ingestion.ometa.ometa_api import OpenMetadata
from metadata.generated.schema.entity.services.connections.metadata.openMetadataConnection import OpenMetadataConnection
from metadata.generated.schema.api.lineage.addLineage import AddLineageRequest
from metadata.generated.schema.type.entityLineage import EntitiesEdge
from metadata.generated.schema.type.entityReference import EntityReference
from metadata.generated.schema.entity.data.table import Table

# 1. Credentials inherited from environment

# 2. Setup Connection (Str casting for Pydantic v2 strictness)
server_config = OpenMetadataConnection(
    hostPort=str(os.environ.get('API_COLLATE_BASE')),
    authProvider="openmetadata",
    securityConfig={"jwtToken": str(os.environ.get('TOKEN'))}
)
metadata = OpenMetadata(server_config)

# 3. Define the FQNs for MovR
users_fqn = "Cockroach_movr.movr.public.users"
rides_fqn = "Cockroach_movr.movr.public.rides"
view_fqn  = "Cockroach_movr.movr.public.customer_summary_view"

def link_entities(source_fqn, target_fqn):
    # 'Table' is now defined and ready to use
    src_entity = metadata.get_by_name(entity=Table, fqn=source_fqn)
    tgt_entity = metadata.get_by_name(entity=Table, fqn=target_fqn)
    
    if src_entity and tgt_entity:
        lineage_edge = AddLineageRequest(
            edge=EntitiesEdge(
                fromEntity=EntityReference(id=src_entity.id, type="table"),
                toEntity=EntityReference(id=tgt_entity.id, type="table")
            )
        )
        metadata.add_lineage(lineage_edge)
        print(f"✅ Linked {source_fqn} -> {target_fqn}")
    else:
        print(f"❌ Could not find entities for {source_fqn} or {target_fqn}")

# 4. Execute
if metadata.health_check():
    # Fetch the full entity with all its 'Relationship' data
    table_entity = metadata.get_by_name(
        entity=Table, 
        fqn="Cockroach_movr.movr.public.rides", 
        fields=["*"]
    )

    print(f"Entity FQN: {table_entity.fullyQualifiedName.root}")
    # In the SDK, 'entityType' is a class-level attribute or found in serviceType
    print(f"Service Type: {table_entity.serviceType.value}")

    # Inspect the 'Owners' relationship (Pydantic v2 uses .root for many fields)
    if table_entity.owners:
        for owner in table_entity.owners.root:
            print(f"Relationship: [Table] --(OwnedBy)--> [User:{owner.name}]")
    else:
        print("Relationship: [Table] --(OwnedBy)--> [None]")

    # Inspect the 'Tags' relationship
    if table_entity.tags:
        for tag in table_entity.tags:
            print(f"Relationship: [Table] --(HasTag)--> [Tag:{tag.tagFQN.root}]")

    # see the raw graph node
    # Convert the Pydantic object back to a raw Dictionary
    raw_ontology_node = table_entity.model_dump()
    print("raw dictionary dump")
    print(json.dumps(raw_ontology_node, indent=2, default=str))

## Lineage & Health Checks

## Service & Glossary Management

## Pipelines & Users

## Discovery

In [ ]:
# Native Python SDK Imports
from metadata.generated.schema.entity.data.table import Table
from metadata.generated.schema.entity.services.databaseService import DatabaseService
from metadata.generated.schema.entity.services.searchService import SearchService
from metadata.generated.schema.entity.data.glossary import Glossary
from metadata.generated.schema.entity.teams.user import User
from metadata.generated.schema.entity.teams.role import Role
from metadata.generated.schema.entity.services.ingestionPipelines.ingestionPipeline import IngestionPipeline

print("✅ SDK Classes imported.")

## Lineage & Health Checks (Python SDK)

In [ ]:
# checkLineage logic
table_fqn = "Cockroach_movr.movr.public.rides"
lineage = metadata.get_lineage_by_name(entity=Table, fqn=table_fqn)
print(f"Lineage for {table_fqn}:")
print(json.dumps(lineage, indent=2, default=str))

In [ ]:
# compare_counts logic
user_list = metadata.list_entities(entity=User, limit=1)
print(f"📊 Database Users (Total): {user_list.paging.total}")

## Service & Glossary Management (Python SDK)

In [ ]:
# getDBService logic
service_name = "Cockroach_movr"
svc = metadata.get_by_name(entity=DatabaseService, fqn=service_name, fields=["owners", "tags"])
if svc:
    print(f"✅ Found Service: {svc.name.root}")
    print(svc.model_dump_json(indent=2))

In [ ]:
# getSearchService logic
service_name = "elasticsearch"
svc = metadata.get_by_name(entity=SearchService, fqn=service_name, fields=["owners", "tags"])
if svc:
    print(f"✅ Found Search Service: {svc.name.root}")
    print(svc.model_dump_json(indent=2))
else:
    print(f"❌ Search Service {service_name} not found.")

In [ ]:
# getGlossary logic
glossary_name = "Business Glossary"
glossary = metadata.get_by_name(entity=Glossary, fqn=glossary_name)
if glossary:
    print(f"✅ Found Glossary: {glossary.name.root}")
    print(glossary.model_dump_json(indent=2))

## Pipelines & Users (Python SDK)

In [ ]:
# getPipelines logic
service_name = "Cockroach_movr"
pipelines = metadata.list_all_entities(entity=IngestionPipeline, fields=["owners"])
filtered = [p for p in pipelines if p.service.name == service_name]

print(f"✅ Found {len(filtered)} pipelines for {service_name}:")
for p in filtered:
    print(f"- {p.name.root} ({p.pipelineType.value})")

In [ ]:
# getOwnerID logic
owner_name = "jason.haugland"
user = metadata.get_by_name(entity=User, fqn=owner_name)
if user:
    print(f"👤 User: {user.name}")
    print(f"🆔 ID: {user.id}")

## Discovery (Python SDK)

In [ ]:
# list_roles logic
roles = metadata.list_entities(entity=Role)
print("Available Roles:")
for r in roles.data:
    print(f"- {r.name.root}: {r.id}")

In [ ]:
# list_services logic
services = metadata.list_entities(entity=DatabaseService)
print("Database Services:")
for s in services.data:
    print(f"- {s.name.root} ({s.serviceType.value})")

In [ ]:
import requests
import os
import json

# 1. Setup
base_url = os.environ.get('API_BASE')
print(f" this is the base url {base_url}")
token = os.environ.get('TOKEN')
fqn = "Cockroach_movr.movr.public.rides" # Double-check this exact string in the UI
print(f" this is the fqn {fqn}")
url = f"{base_url}/tables/name/{fqn}?fields=*"
headers = {"Authorization": f"Bearer {token}"}
print(f" this is the url {url}")
print(f" this is the headers {headers}")

# 2. Call the API
response = requests.get(url, headers=headers)

if response.status_code == 200:
    raw_node = response.json()
    
    # 3. Safe Exploration
    name = raw_node.get('name', 'Unknown')
    cols = [c.get('name') for c in raw_node.get('columns', [])]
    
    print(f"✅ Entity Found: {name}")
    print(f"🔹 Columns: {', '.join(cols)}")
    print(f"🔹 Version: {raw_node.get('version')}")
    
else:
    print(f"❌ API Error {response.status_code}: {response.text}")

In [ ]:
import requests
import os

# 1. Setup based on your vars
base_url = os.environ.get('API_BASE') # https://demo.getcollate.io/api/v1
token = os.environ.get('TOKEN')
headers = {"Authorization": f"Bearer {token}"}

print(f"🔍 Searching for tables in the 'public' schema at: {base_url}")

# 2. Search for all tables in the 'public' schema
search_url = f"{base_url}/search/query?q=databaseSchema.name:public&index=table_search_index"
res = requests.get(search_url, headers=headers)

if res.status_code == 200:
    hits = res.json().get('hits', {}).get('hits', [])
    print(f"✅ Found {len(hits)} entities.\n")
    for hit in hits:
        fqn = hit['_source']['fullyQualifiedName']
        print(f"📍 EXACT FQN: {fqn}")
else:
    print(f"❌ Search Failed: {res.text}")